# Spark Structured Streaming — 实时消费 Kafka 数据

**流程：** Kafka `tlc-trips` topic → Spark Structured Streaming → 实时聚合结果

运行前确认：
1. Kafka broker 正在运行（终端 1）
2. topic `tlc-trips` 已创建
3. notebook 02 的 Producer 准备好随时发送数据

## 1. 初始化 SparkSession（带 Kafka 连接器）

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, IntegerType

spark = (
    SparkSession.builder
    .appName("TLC-Streaming")
    .config("spark.sql.shuffle.partitions", "4")
    # Kafka 连接器，Spark 会自动下载这个 jar 包
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.1")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")
print("SparkSession ready.")

:: loading settings :: url = jar:file:/Users/lujiayi/anaconda3/envs/tlc-spark/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/lujiayi/.ivy2/cache
The jars for the packages stored in: /Users/lujiayi/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-365aa337-31da-411c-ab08-46acd79ea939;1.0
	confs: [default]


	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central


	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central


	found commons-logging#commons-logging;1.1.3 in local-m2-cache
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 854ms :: artifacts dl 24ms
	:: modules in use:
	com.google.code.findbugs#jsr305;3.0.0 from central in [default]
	commons-logging#commons-logging;1.1.3 from local-m2-cache in [default]
	org.apache.commons#commons-pool2;2.11.1 from central in [default]
	org.apache.hadoop#hadoop-client-api;3.3.4 from central in [default]
	org.apache.hadoop#hadoop-client-runtime;3.3.4 from central in [default]
	org.apache.kafka#kafka-clients;3.4.1 from central in [default]
	org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.1 from central in [default]
	org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.1 from central in [default]
	org.lz4#lz4-java;1.8.0 from central in [default]
	org.slf4j#slf4j-api;2.0.7 from central in [default]
	org.xerial.snappy#snappy-java;1.1.10.3 from central in [default]
	-

26/10/19 00:04:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark version: 3.5.1
SparkSession ready.


## 2. 从 Kafka 读取数据流

In [2]:
# readStream：持续监听，不是一次性读取
raw_stream = (
    spark.readStream
    .format("kafka")
    .option("kafka.bootstrap.servers", "localhost:9092")
    .option("subscribe", "tlc-trips")
    .option("startingOffsets", "earliest")  # 从头开始读，包含之前发的 500 条
    .load()
)

print("Stream schema:")
raw_stream.printSchema()

Stream schema:
root
 |-- key: binary (nullable = true)
 |-- value: binary (nullable = true)
 |-- topic: string (nullable = true)
 |-- partition: integer (nullable = true)
 |-- offset: long (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- timestampType: integer (nullable = true)



## 3. 解析 JSON 消息

In [ ]:
# Schema matches what 08_kafka_producer sends (ODS fields, not DWD-derived fields)
trip_schema = StructType([
    StructField("VendorID",        IntegerType(), True),
    StructField("pickup_datetime", StringType(),  True),
    StructField("dropoff_datetime",StringType(),  True),
    StructField("PULocationID",    IntegerType(), True),
    StructField("DOLocationID",    IntegerType(), True),
    StructField("passenger_count", IntegerType(), True),
    StructField("trip_distance",   DoubleType(),  True),
    StructField("fare_amount",     DoubleType(),  True),
    StructField("tip_amount",      DoubleType(),  True),
    StructField("total_amount",    DoubleType(),  True),
    StructField("payment_type",    IntegerType(), True),
    StructField("source_type",     StringType(),  True),
    StructField("event_timestamp", StringType(),  True),
])

# Parse JSON, cast timestamps, apply streaming-side cleaning + derive trip_duration_min
# Producer sends ODS (uncleaned) → consumer cleans independently → reconciliation is meaningful
parsed = (
    raw_stream
    .select(F.from_json(F.col("value").cast("string"), trip_schema).alias("data"))
    .select("data.*")
    .withColumn("event_timestamp",   F.to_timestamp("event_timestamp"))
    .withColumn("pickup_datetime",   F.to_timestamp("pickup_datetime"))
    .withColumn("dropoff_datetime",  F.to_timestamp("dropoff_datetime"))
    .withColumn("trip_duration_min",
        F.round(
            (F.unix_timestamp("dropoff_datetime") - F.unix_timestamp("pickup_datetime")) / 60, 2))
    # Streaming-side cleaning (mirrors offline DWD cleaning for comparable results)
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
    .filter(F.col("trip_duration_min").between(1, 180))
)

print("Parsed schema:")
parsed.printSchema()


## 4. 实时聚合 — 按时段统计行程量和平均票价

In [4]:
import os

# 用 pickup_datetime（事件时间）做水印，允许最多 2 分钟的迟到数据
# 按上车区域统计每 5 分钟内的叫车量
agg_stream = (
    parsed
    .withWatermark("pickup_datetime", "2 minutes")
    .groupBy(
        F.window("pickup_datetime", "5 minutes"),  # 5 分钟滚动窗口
        "PULocationID"
    )
    .agg(
        F.count("*").alias("trip_count"),
        F.round(F.avg("fare_amount"), 2).alias("avg_fare"),
        F.round(F.avg("trip_duration_min"), 1).alias("avg_duration_min")
    )
    .select(
        F.col("window.start").alias("window_start"),
        F.col("window.end").alias("window_end"),
        "PULocationID",
        "trip_count",
        "avg_fare",
        "avg_duration_min"
    )
)

print("Aggregation stream defined.")

Aggregation stream defined.


## 5. 启动流处理 — 双写（Parquet + 控制台）\n\n使用 `foreachBatch` 将每个 micro-batch 的聚合结果同时写入 Parquet 文件和控制台，供下游离线对账使用。

In [ ]:
CHECKPOINT_PATH = os.path.expanduser("~/tlc-spark-pipeline/data/streaming/checkpoints/zone-demand")
STREAM_OUT      = os.path.expanduser("~/tlc-spark-pipeline/data/streaming/output")

def write_batch(batch_df, batch_id):
    """每个 micro-batch：写入 Parquet + 打印控制台"""
    if not batch_df.isEmpty():
        batch_df.write.mode("append").parquet(STREAM_OUT)
        print(f"\n--- Batch {batch_id} ---")
        batch_df.orderBy("window_start", "PULocationID").show(20, truncate=False)

# outputMode="append": 每个时间窗口在水印推进后只输出一次（窗口关闭后），
# 避免 update 模式下同一窗口被多次写入 Parquet 导致重复累加
query = (
    agg_stream
    .writeStream
    .outputMode("append")
    .foreachBatch(write_batch)
    .option("checkpointLocation", CHECKPOINT_PATH)
    .trigger(processingTime="10 seconds")
    .start()
)

print("Streaming query started. 等待数据...")
print("结果每 10 秒写入 Parquet 并显示。")

query.awaitTermination(timeout=90)
print(f"\nStreaming 结束。输出写入: {STREAM_OUT}")


## 6. 离线 vs 实时对账\n\n读取 Streaming 写出的 Parquet，与 ADS 层离线 trip_count 对比，计算差异率。\n\n差异来源：Streaming 只发送了 5000 条（1月前 ~57 分钟），离线覆盖 Q1 全量；对账时按时间窗口过滤对齐。

In [ ]:
import os

STREAM_OUT = os.path.expanduser("~/tlc-spark-pipeline/data/streaming/output")
DWD_FACT   = os.path.expanduser("~/tlc-spark-pipeline/data/warehouse/dwd/fact_trips")

# 实时结果（streaming 写出的 Parquet，按 PULocationID 汇总）
streaming_df = spark.read.parquet(STREAM_OUT)
streaming_total = streaming_df.agg({"trip_count": "sum"}).collect()[0][0]

# 对应时间窗口的离线结果
# 注意：streaming 只发送 yellow 出租车数据，对账时离线也只取 yellow
min_ws = streaming_df.agg({"window_start": "min"}).collect()[0][0]
max_we = streaming_df.agg({"window_end":   "max"}).collect()[0][0]

offline_df = spark.read.parquet(DWD_FACT)
offline_total = (
    offline_df
    .filter(offline_df["source_type"] == "yellow")   # 与 streaming 保持同源
    .filter(
        (offline_df["pickup_datetime"] >= min_ws) &
        (offline_df["pickup_datetime"] <  max_we)
    )
    .count()
)

diff     = abs(streaming_total - offline_total)
diff_pct = diff / offline_total * 100 if offline_total > 0 else 0

print(f"对账窗口: {min_ws}  ~  {max_we}")
print(f"Streaming 统计量 (yellow): {streaming_total:,}")
print(f"离线同窗口行数  (yellow): {offline_total:,}")
print(f"差异量:                   {diff:,}  ({diff_pct:.2f}%)")
print(f"\n差异来源：两侧都只统计 yellow，但清洗口径略有不同（streaming 侧按阈值过滤，离线侧经过额外的速度/时长过滤）")
print(f"\n{'✅ 差异率低于 10%' if diff_pct < 10 else '⚠️  差异率超过 10%，需排查'}")


In [7]:
query.stop()
spark.stop()
print("Spark stopped.")

Spark stopped.
